## Streaming & Batch in LangChain
- **Streaming** — get the response word by word (like ChatGPT)
- **Batch** — send multiple queries at once in parallel

Both are built-in methods: `.stream()` and `.batch()`

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model='openai/gpt-oss-20b')

### Basic Streaming — `.stream()`
Returns the response chunk by chunk. Each chunk is an `AIMessageChunk` object.

In [ ]:
from langchain_core.messages import HumanMessage

# stream karte hain — response ek ek karke aayega
for chunk in model.stream([HumanMessage(content='Python ke 5 popular libraries batao')]):
    print(chunk.content, end='', flush=True)

### Inspecting stream chunks
We can collect chunks and see what each one contains.

In [ ]:
# chunk ki details dekhte hain
chunks = []
for chunk in model.stream([HumanMessage(content='Namaste bolke apna intro do')]):
    chunks.append(chunk)
    print(f'Chunk: "{chunk.content}"')

print(f'\nTotal chunks received: {len(chunks)}')

### Merging chunks
Combine all chunks into a single complete message using the `+` operator.

In [ ]:
# chunks ko merge karke full response banana
full_response = chunks[0]
for c in chunks[1:]:
    full_response = full_response + c

print('Full merged response:')
print(full_response.content)

### Async Streaming — `astream()`
Use this in async environments like FastAPI or async functions.

In [ ]:
import asyncio

async def stream_response():
    async for chunk in model.astream([HumanMessage(content='ek chhota sa joke sunao')]):
        print(chunk.content, end='', flush=True)

await stream_response()

### Batch Processing — `.batch()`
Send multiple queries at once. Runs in parallel internally, so it's faster than calling `.invoke()` one by one.

In [ ]:
# ek saath 3 queries bhejte hain
queries = [
    [HumanMessage(content='Python kya hai? 1 line mein batao')],
    [HumanMessage(content='JavaScript kya hai? 1 line mein batao')],
    [HumanMessage(content='Rust kya hai? 1 line mein batao')]
]

responses = model.batch(queries)

for i, resp in enumerate(responses):
    print(f'{i+1}. {resp.content}\n')

### Batch with max_concurrency
Limit the number of parallel requests to avoid rate limits.

In [ ]:
# rate limit se bachne ke liye max_concurrency use karte hain
queries = [
    [HumanMessage(content='HTML kya hai?')],
    [HumanMessage(content='CSS kya hai?')],
    [HumanMessage(content='React kya hai?')],
    [HumanMessage(content='Node.js kya hai?')],
    [HumanMessage(content='Django kya hai?')]
]

# max 2 parallel requests
results = model.batch(queries, config={'max_concurrency': 2})

for r in results:
    print(r.content)
    print('---')

### Async Batch — `abatch()`
Async version of batch for use in async code.

In [ ]:
async def run_batch():
    queries = [
        [HumanMessage(content='Java ka full form kya hai?')],
        [HumanMessage(content='SQL ka full form kya hai?')]
    ]
    results = await model.abatch(queries)
    for r in results:
        print(r.content)

await run_batch()

### Streaming with a chain
Streaming works on the entire chain, not just the model. If you have a `prompt | model` chain, you can stream it.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Tum ek {role} ho. Short mein jawab do.'),
    ('human', '{question}')
])

chain = prompt | model

# chain pe streaming
for chunk in chain.stream({'role': 'doctor', 'question': 'sar dard ho toh kya karna chahiye?'}):
    print(chunk.content, end='', flush=True)

### Batch with a chain
Send multiple different inputs through the same chain at once.

In [ ]:
# chain pe batch
inputs = [
    {'role': 'teacher', 'question': 'gravity kya hai?'},
    {'role': 'chef', 'question': 'biryani kaise banti hai?'},
    {'role': 'coder', 'question': 'API kya hoti hai?'}
]

results = chain.batch(inputs)

for i, r in enumerate(results):
    print(f'--- Response {i+1} ---')
    print(r.content)
    print()

### Quick comparison

| Method | What it does | When to use |
|--------|-------------|-------------|
| `.invoke()` | Single query, full response at once | Normal use |
| `.stream()` | Single query, response chunk by chunk | Chatbot UI, real-time display |
| `.batch()` | Multiple queries in parallel | Bulk processing |
| `.ainvoke()` / `.astream()` / `.abatch()` | Async versions | FastAPI, async apps |